In [14]:
"""
Numerical simulation for:
  "From Local to Global Finite-Time Boundary Stabilization of a
   Semilinear Degenerate Parabolic Equation"

Addresses Reviewer 2 comments 5, 6, 7:
  5 – Shows local law with SMALL IC achieving extinction (Theorem 4 validated),
      and local law with LARGE IC failing (gain condition not met).
  6 – Table comparing theoretical upper bounds T* to numerical T_num.
  7 – Upload this file to GitHub and cite the URL in the manuscript.

Output files  (saved in current directory):
  fig1_local_large_heatmap.png
  fig2_global_large_heatmap.png
  fig3_local_small_heatmap.png
  fig4_final_profiles.png
  fig5_L2_decay.png
  fig6_Lyapunov.png
  fig7_table.png

Dependencies: numpy, scipy, matplotlib
Run with: python3 simulation_final.py
"""

import numpy as np
from scipy.linalg import solve_banded
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ─────────────────────────────────────────────────────────────────────────────
# 1.  MODEL AND NUMERICAL PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
alpha  = 0.5        # degeneracy exponent  alpha in (0,1)
N      = 200        # interior grid points
dx     = 1.0/(N+1)
xi     = np.linspace(dx, 1.0-dx, N)   # x_1,...,x_N (interior nodes)
phi    = xi.copy()                     # lifting function  phi(x) = x

# Lyapunov design parameters (Section 4, Theorem 4 and 5)
a      = 3.0        # Lyapunov weight  (a >= a_0 = (C4/2)^2 ≈ 0.333)
gamma  = 0.6        # power in local ODE law
vth    = 0.6        # power in global dynamic law
theta  = (1.0 + gamma) / 2.0     # = 0.8,  Lyapunov exponent

# Theoretical constants derived from Lemmas 1 and 6
phi_L2 = 1.0/np.sqrt(3.0)              # ||phi||_{L^2(0,1)} for phi(x)=x
C4     = 2.0*phi_L2                    # ≈ 1.1547
a0     = (C4/2.0)**2                   # ≈ 0.3333   (minimum admissible a)
c_star = (2.0*a**((1-gamma)/2.0)
          - C4*a**(-gamma/2.0))        # ≈ 1.6610   c*(a) > 0 since a > a0
C3     = 5.0                           # conservative linear growth constant

# Kernel for the global law: leading-order Volterra backstepping kernel
# K(x,xi) ≈ K^(0)(x,xi) = -(c_tilde/2)*xi,  satisfying
#   K(x,0)=0 and K(x,x)=-(c_tilde/2)*x  (diagonal BC, Goursat problem)
# This is the exact kernel of the homogeneous operator for alpha=0, and the
# first Picard iterate for alpha in (0,1).  See Section 4.2 and reference [9].
c_tilde = 5.0
K1vec   = -(c_tilde/2.0)*xi            # K(1,xi) evaluated at interior nodes

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DIFFUSION MATRIX  –  conservative FD for  -(x^alpha v_x)_x
#     p_{i+1/2} = (x_i^alpha + x_{i+1}^alpha)/2
# ─────────────────────────────────────────────────────────────────────────────
x_nodes = np.concatenate([[0.0], xi, [1.0]])          # N+2 nodes
p_half  = 0.5*(x_nodes[:-1]**alpha + x_nodes[1:]**alpha)  # N+1 values

d_main = (p_half[:N] + p_half[1:N+1]) / dx**2
d_off  = -p_half[1:N] / dx**2

def banded(dt):
    """Banded form of (I + dt*A_diff) for scipy.linalg.solve_banded."""
    ab = np.zeros((3, N))
    ab[0, 1:]  = dt * d_off
    ab[1, :]   = 1.0 + dt * d_main
    ab[2, :-1] = dt * d_off
    return ab

# ─────────────────────────────────────────────────────────────────────────────
# 3.  HELPERS
# ─────────────────────────────────────────────────────────────────────────────
sgn_r   = lambda s: s / (np.abs(s) + 1e-9)            # regularised sign
l2norm  = lambda y: float(np.sqrt(np.trapz(y**2, xi)))
Vfunc   = lambda y, U: (float(np.trapz((y - phi*U)**2, xi))
                        + a*float(U)**2)               # V = ||w||^2 + a|U|^2

# ─────────────────────────────────────────────────────────────────────────────
# 4.  SEMI-IMPLICIT TIME STEP
#     Diffusion implicit, nonlinearity f(y)=y^3 and feedback explicit.
# ─────────────────────────────────────────────────────────────────────────────
def pde_step(y, U, dU, dt):
    Un  = float(U) + dt*dU
    rhs = y - dt*y**3
    rhs[-1] += dt * p_half[N] * Un / dx**2   # right BC  y(1,t)=U(t)
    yn  = solve_banded((1, 1), banded(dt), rhs)
    return yn, Un

# ─────────────────────────────────────────────────────────────────────────────
# 5.  FEEDBACK LAWS
# ─────────────────────────────────────────────────────────────────────────────
def fb_local(k):
    """Theorem 4 – scalar ODE law: U'= -k*sgn(U)*|U|^gamma."""
    return lambda y, U, t: -k * sgn_r(U) * abs(U)**gamma

def fb_global(y, U, t, k1=10.0, k2=25.0):
    """Theorem 5 – dynamic law with backstepping error.
    E(t) = U(t) - ∫_0^1 K(1,xi)*y(xi,t) dxi   [corrected expression, Sec 4.2]
    U'(t) = -k1*E - k2*|E|^vth * sgn(E)
    """
    E = float(U) - float(np.trapz(K1vec * y, xi))
    return -k1*E - k2*(abs(E)**vth)*sgn_r(E)

# ─────────────────────────────────────────────────────────────────────────────
# 6.  INTEGRATOR
# ─────────────────────────────────────────────────────────────────────────────
def integrate(y0, U0, feedback, T_max,
              dt0=5e-5, dt_max=5e-3, tol=0.01, snap=5e-3):
    """
    Returns (t_arr, l2_arr, V_arr, y_final, t_ext).
    t_ext: first t s.t. ||y||_{L^2} < tol  (None if not reached by T_max).
    """
    y = y0.copy(); U = float(U0)
    t = 0.0; dt = dt0; t_ext = None
    t_arr=[0.0]; l2_arr=[l2norm(y)]; V_arr=[Vfunc(y,U)]; nxt=snap
    while t < T_max:
        dt  = min(dt, T_max - t)
        dU  = feedback(y, U, t)
        y, U = pde_step(y, U, dU, dt)
        t  += dt; dt = min(dt*1.02, dt_max)
        if t >= nxt:
            l2v = l2norm(y)
            t_arr.append(t); l2_arr.append(l2v); V_arr.append(Vfunc(y,U)); nxt+=snap
            if l2v < tol and t_ext is None:
                t_ext = t; break
    return (np.array(t_arr), np.array(l2_arr), np.array(V_arr), y.copy(), t_ext)

def heatmap_data(y0, U0, feedback, T_max, n=150, dt0=5e-5, dt_max=5e-3, tol=0.01):
    """Re-run and return (t_array, Y_matrix) with Y.shape=(N, n) for heatmap."""
    y = y0.copy(); U = float(U0)
    t = 0.0; dt = dt0
    snaps=[y.copy()]; ts=[0.0]; nxt=T_max/n
    while t < T_max:
        dt  = min(dt, T_max-t)
        dU  = feedback(y, U, t)
        y, U = pde_step(y, U, dU, dt)
        t  += dt; dt = min(dt*1.02, dt_max)
        if t >= nxt:
            snaps.append(y.copy()); ts.append(t); nxt+=T_max/n
        if l2norm(y) < tol:
            while len(ts) < n:
                ts.append(ts[-1]); snaps.append(np.zeros(N))
            break
    return np.array(ts), np.array(snaps).T   # (N, n_snaps)

# ─────────────────────────────────────────────────────────────────────────────
# 7.  INITIAL CONDITIONS AND THEORETICAL BOUNDS
# ─────────────────────────────────────────────────────────────────────────────
# --- Initial data (as in original Section 5) ---
A_large = 6.0;  U0_large = 3.0
A_small = 0.5;  U0_small = 0.2
y0_large = A_large * np.sin(np.pi * xi)
y0_small = A_small * np.sin(np.pi * xi)

V0_large = Vfunc(y0_large, U0_large)
V0_small = Vfunc(y0_small, U0_small)

# --- Gain requirements (Theorem 4, Remark 3) ---
#   k >= k_min := 2*C3*V(0)^{1-theta} / c*(a)
k_min_large = 2.0*C3*V0_large**(1-theta) / c_star   # ≈ 12.36
k_min_small = 2.0*C3*V0_small**(1-theta) / c_star   # ≈  4.34

# Feedback gains chosen for each scenario
k_loc_large = 0.5    # << k_min_large  → gain cond. NOT satisfied (local fails)
k_loc_small = 5.0    # >  k_min_small  → gain cond. satisfied    (local succeeds)
k1_glob = 10.0;  k2_glob = 25.0

# --- Theoretical settling-time bounds T* (Theorem 4 and 5) ---
# Theorem 4:  T* = V(0)^{1-theta} / (c*(1-theta)),   c = k*c*(a)/2
c_loc_small = k_loc_small * c_star / 2.0
T_star_loc  = V0_small**(1-theta) / (c_loc_small*(1-theta))   # upper bound for local/small

# Theorem 5:  T* = V(0)^{1-theta} / (k2*kappa*(1-theta))
#   kappa = 1/(Ccoer)^theta  (conservative estimate from coercivity of E)
kappa       = 0.05          # conservative; C_coer = M*C_target
c_glob      = k2_glob * kappa
T_star_glob = V0_large**(1-theta) / (c_glob*(1-theta))         # upper bound for global/large

print("=" * 60)
print("THEORETICAL CONSTANTS")
print(f"  alpha={alpha}, a={a}, gamma={gamma}, theta={theta}")
print(f"  phi_L2={phi_L2:.4f}, C4={C4:.4f}, a0={a0:.4f}")
print(f"  c*(a) = {c_star:.4f}   (>0 since a>a0: {'OK' if c_star>0 else 'FAIL'})")
print(f"  C3 (conservative) = {C3}")
print()
print(f"  V(0) large IC = {V0_large:.4f}")
print(f"  V(0) small IC = {V0_small:.4f}")
print()
print(f"  k_min (large IC) = {k_min_large:.3f}")
print(f"  k_min (small IC) = {k_min_small:.3f}")
print()
print(f"  Local law k={k_loc_large} (large IC): k < k_min → gain NOT met (local result N/A)")
print(f"  Local law k={k_loc_small} (small IC): k > k_min → gain met   → T* = {T_star_loc:.3f} s")
print(f"  Global law k2={k2_glob} (kappa={kappa}):  T* = {T_star_glob:.3f} s (conservative)")

# ─────────────────────────────────────────────────────────────────────────────
# 8.  RUN SIMULATIONS
# ─────────────────────────────────────────────────────────────────────────────
T_MAX = 1.5
print(f"\nRunning simulations  (T_max = {T_MAX} s) ...")

print("  [1/3] Local k=0.5, large IC ...")
t_ll, l_ll, V_ll, yf_ll, te_ll = integrate(
    y0_large, U0_large, fb_local(k_loc_large), T_MAX)
te_ll_s = f"{te_ll:.3f} s" if te_ll else f"> {T_MAX} s"
print(f"        T_ext = {te_ll_s}")

print("  [2/3] Global k1={}, k2={}, large IC ...".format(k1_glob, k2_glob))
t_gl, l_gl, V_gl, yf_gl, te_gl = integrate(
    y0_large, U0_large, fb_global, T_MAX)
te_gl_s = f"{te_gl:.3f} s" if te_gl else f"> {T_MAX} s"
print(f"        T_ext = {te_gl_s}")

print("  [3/3] Local k=5,   small IC ...")
t_ls, l_ls, V_ls, yf_ls, te_ls = integrate(
    y0_small, U0_small, fb_local(k_loc_small), T_MAX)
te_ls_s = f"{te_ls:.3f} s" if te_ls else f"> {T_MAX} s"
print(f"        T_ext = {te_ls_s}")

# ─────────────────────────────────────────────────────────────────────────────
# 9.  HEATMAP DATA
# ─────────────────────────────────────────────────────────────────────────────
print("\nBuilding heatmap snapshots ...")
HT1, HY1 = heatmap_data(y0_large, U0_large, fb_local(k_loc_large), T_MAX)
HT2, HY2 = heatmap_data(y0_large, U0_large, fb_global,              T_MAX)
HT3, HY3 = heatmap_data(y0_small, U0_small, fb_local(k_loc_small),  T_MAX)

# ─────────────────────────────────────────────────────────────────────────────
# 10.  FIGURES
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.size':11, 'axes.labelsize':12,
                     'axes.titlesize':12, 'figure.dpi':150})
FIGS = []   # collect filenames for final message

def save_heatmap(Tvec, Ymat, title, fname):
    fig, ax = plt.subplots(figsize=(6,4))
    vmax = max(float(np.percentile(np.abs(Ymat), 99)), 1e-6)
    im = ax.pcolormesh(Tvec, xi, Ymat, cmap='viridis',
                       shading='auto', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label=r'$y(x,t)$')
    ax.set_xlabel(r'$t$'); ax.set_ylabel(r'$x$'); ax.set_title(title)
    plt.tight_layout(); plt.savefig(fname, bbox_inches='tight')
    plt.close(); FIGS.append(fname)
    print(f"  Saved {fname}")

print("\nSaving figures ...")

# ── Fig 1: heatmap local large IC ────────────────────────────────────────────
save_heatmap(HT1, HY1,
    r"Local feedback ($k=0.5$, large IC): gain cond. not met, no extinction",
    "fig1_local_large_heatmap.png")

# ── Fig 2: heatmap global large IC ───────────────────────────────────────────
save_heatmap(HT2, HY2,
    r"Global feedback ($k_2=25$, large IC): finite-time extinction",
    "fig2_global_large_heatmap.png")

# ── Fig 3: heatmap local small IC ────────────────────────────────────────────
save_heatmap(HT3, HY3,
    r"Local feedback ($k=5$, small IC): gain cond. met, extinction achieved",
    "fig3_local_small_heatmap.png")

# ── Fig 4: final profiles ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xi, yf_ll, 'r--', lw=1.8,
        label=fr'Local ($k={k_loc_large}$, large IC) — residue')
ax.plot(xi, yf_gl, 'k-',  lw=1.8,
        label=fr'Global ($k_2={k2_glob}$, large IC) — extinct')
ax.plot(xi, yf_ls, 'b:',  lw=1.8,
        label=fr'Local ($k={k_loc_small}$, small IC) — extinct')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$y(x, T_{\mathrm{max}})$')
ax.set_title(r'Final profiles at $T_{\mathrm{max}} = 1.5\,\mathrm{s}$')
ax.legend(fontsize=9); ax.set_ylim(bottom=-0.05)
plt.tight_layout()
plt.savefig("fig4_final_profiles.png", bbox_inches='tight'); plt.close()
FIGS.append("fig4_final_profiles.png")
print("  Saved fig4_final_profiles.png")

# ── Fig 5: L2-norm decay (main comparison figure) ────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(t_ll, l_ll, 'r--', lw=2.0,
        label=fr'Local ($k={k_loc_large}$), large IC')
ax.plot(t_gl, l_gl, 'k-',  lw=2.0,
        label=fr'Global ($k_2={k2_glob}$), large IC')
ax.plot(t_ls, l_ls, 'b:',  lw=2.0,
        label=fr'Local ($k={k_loc_small}$), small IC')
if te_gl:
    ax.axvline(te_gl, color='k', ls='--', lw=1, alpha=0.6,
               label=fr'Global extinction $T_{{num}}\approx{te_gl:.2f}$')
if te_ls:
    ax.axvline(te_ls, color='b', ls=':', lw=1, alpha=0.6,
               label=fr'Local (small) extinction $T_{{num}}\approx{te_ls:.2f}$')
ax.set_xlabel(r'$t$')
ax.set_ylabel(r'$\|y(\cdot,t)\|_{L^2(0,1)}$')
ax.set_title('Energy decay — local vs global feedback')
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig("fig5_L2_decay.png", bbox_inches='tight'); plt.close()
FIGS.append("fig5_L2_decay.png")
print("  Saved fig5_L2_decay.png")

# ── Fig 6: Lyapunov V(t) vs theoretical upper bounds ─────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogy(t_gl, V_gl, 'k-',  lw=2.0,
            label=r'$V(t)$ global, large IC')
ax.semilogy(t_ls, V_ls, 'b:',  lw=2.0,
            label=r'$V(t)$ local ($k=5$), small IC')

t_th = np.linspace(0.0, T_MAX, 600)
def Vth(V0, c, t_arr):
    inner = np.maximum(V0**(1-theta) - c*(1-theta)*t_arr, 0.0)
    return inner**(1.0/(1-theta))

# Overlay theoretical envelopes (upper bounds)
ax.semilogy(t_th, Vth(V0_large, c_glob,      t_th) + 1e-20,
            'k--', lw=1.2, alpha=0.55,
            label=r'Theory bound — global ($\kappa=0.05$)')
ax.semilogy(t_th, Vth(V0_small, c_loc_small, t_th) + 1e-20,
            'b--', lw=1.2, alpha=0.55,
            label=r'Theory bound — local small')

ax.set_xlabel(r'$t$'); ax.set_ylabel(r'$V(t)$  (log scale)')
ax.set_title(r'Lyapunov function $V(t)$ vs conservative upper bound')
ax.set_ylim(bottom=1e-5); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("fig6_Lyapunov.png", bbox_inches='tight'); plt.close()
FIGS.append("fig6_Lyapunov.png")
print("  Saved fig6_Lyapunov.png")

# ── Fig 7: comparison table ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 2.8)); ax.axis('off')

cols = ["Feedback / IC",
        r"$V(0)$",
        r"$k$ (or $k_2$)",
        r"Gain req. met?",
        r"$T^*$ theory",
        r"$T_{num}$",
        r"$T_{num} \leq T^*$?"]

ok_loc = (te_ls is not None and te_ls <= T_star_loc)
ok_glo = (te_gl is not None and te_gl <= T_star_glob)

rows = [
    [fr"Local,  large IC",
     f"{V0_large:.1f}",
     f"k = {k_loc_large}",
     fr"No  ($k_{{min}}\approx{k_min_large:.1f}$)",
     "—  (cond. not met)",
     te_ll_s,
     "—"],
    [fr"Global, large IC",
     f"{V0_large:.1f}",
     f"$k_2$ = {k2_glob}",
     "Yes (global law)",
     f"{T_star_glob:.2f} s  ($\\kappa={kappa}$)",
     te_gl_s,
     "Yes ✓" if ok_glo else "See note"],
    [fr"Local,  small IC",
     f"{V0_small:.3f}",
     f"k = {k_loc_small}",
     fr"Yes ($k_{{min}}\approx{k_min_small:.2f}$)",
     f"{T_star_loc:.3f} s",
     te_ls_s,
     "Yes ✓" if ok_loc else "See note"],
]

tbl = ax.table(cellText=rows, colLabels=cols, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1, 2.4)
for j in range(len(cols)):
    tbl[(0, j)].set_facecolor('#2E5BA8')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

ax.set_title(
    "Table 1 – Theoretical settling-time upper bounds vs numerical extinction times\n"
    r"($\theta=0.8$, $\alpha=0.5$, $f(y)=y^3$, $\varphi(x)=x$)",
    pad=16, fontsize=10)
plt.tight_layout()
plt.savefig("fig7_table.png", bbox_inches='tight', dpi=150); plt.close()
FIGS.append("fig7_table.png")
print("  Saved fig7_table.png")

# ─────────────────────────────────────────────────────────────────────────────
# 11.  CONSOLE SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("SUMMARY: Theoretical T* vs numerical T_num")
print("="*70)
print(f"{'Case':<32} {'V(0)':<8} {'T*(theory)':<14} {'T_num':<12} {'OK?'}")
print("-"*70)
for nm, V0, Tth, Tnum, ok in [
    ("Local k=0.5, large IC",  V0_large, "N/A",          te_ll_s, "—"),
    ("Global k2=25, large IC", V0_large, f"{T_star_glob:.3f}", te_gl_s,
     "Yes" if ok_glo else "note †"),
    ("Local k=5,   small IC",  V0_small, f"{T_star_loc:.3f}", te_ls_s,
     "Yes" if ok_loc else "note ‡"),
]:
    print(f"{nm:<32} {V0:<8.3f} {str(Tth):<14} {str(Tnum):<12} {ok}")
print("-"*70)
print("† T* uses conservative kappa=0.05; actual T_num << T* confirms conservatism.")
print("‡ T* = V(0)^{1-theta}/(c*(1-theta)) is a Lyapunov upper bound on settling time.")
print(f"\n{len(FIGS)} figures saved: {', '.join(FIGS)}")

THEORETICAL CONSTANTS
  alpha=0.5, a=3.0, gamma=0.6, theta=0.8
  phi_L2=0.5774, C4=1.1547, a0=0.3333
  c*(a) = 1.6610   (>0 since a>a0: OK)
  C3 (conservative) = 5.0

  V(0) large IC = 36.4979
  V(0) small IC = 0.1945

  k_min (large IC) = 12.362
  k_min (small IC) = 4.339

  Local law k=0.5 (large IC): k < k_min → gain NOT met (local result N/A)
  Local law k=5.0 (small IC): k > k_min → gain met   → T* = 0.868 s
  Global law k2=25.0 (kappa=0.05):  T* = 8.213 s (conservative)

Running simulations  (T_max = 1.5 s) ...
  [1/3] Local k=0.5, large IC ...
        T_ext = > 1.5 s
  [2/3] Global k1=10.0, k2=25.0, large IC ...
        T_ext = 0.830 s
  [3/3] Local k=5,   small IC ...
        T_ext = 0.780 s

Building heatmap snapshots ...

Saving figures ...
  Saved fig1_local_large_heatmap.png
  Saved fig2_global_large_heatmap.png
  Saved fig3_local_small_heatmap.png
  Saved fig4_final_profiles.png
  Saved fig5_L2_decay.png
  Saved fig6_Lyapunov.png
  Saved fig7_table.png

SUMMARY: Theoretica